# Inferno on GPU

Runs all six rungs plus vLLM on CUDA and writes everything to `results/gpu/`.

**Before you start — in the right-hand settings panel:**

1. **Accelerator → GPU T4 x2.** Not P100: vLLM needs CUDA compute capability >= 7.0,
   a T4 is 7.5 and a P100 is 6.0. Picking P100 costs you the vLLM comparison, which is
   the reason for coming to GPU at all.
2. **Internet → On.** Needed to pip install and to pull the model. Both this and the GPU
   require phone verification on your Kaggle account.

Budget: ~30 min for the rungs, ~20 min for vLLM (most of it the first install).
You have 30 GPU-hours/week, so quota is not the constraint — but GPU hours burn while
a GPU session is attached, so debug with the accelerator **off** and switch it on to measure.


## 0 · Check the GPU before spending anything on it


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU attached - set Accelerator in the sidebar'
cap = torch.cuda.get_device_capability(0)
print('device  :', torch.cuda.get_device_name(0))
print('capability:', cap)
if cap[0] < 7:
    print('\n*** This GPU cannot run vLLM (needs compute capability >= 7.0).')
    print('*** Switch the accelerator to T4 x2 and restart, or the rungs will run')
    print('*** but the vLLM comparison will not.')
else:
    print('\nOK for vLLM.')


## 1 · Get the code


In [ ]:
REPO = 'https://github.com/aditicodes7/inferno.git'   # <- change if your remote differs
import os, shutil
if os.path.exists('/kaggle/working/inferno'):
    shutil.rmtree('/kaggle/working/inferno')
!git clone -q $REPO /kaggle/working/inferno
%cd /kaggle/working/inferno
!git log --oneline | head -3


If the repo is **private**, `git clone` will fail. Either make it public, or add a
GitHub personal access token as a Kaggle Secret and clone with
`https://<token>@github.com/<you>/inferno.git`.


## 2 · Dependencies

Kaggle ships torch already, so this only tops up what's missing.


In [ ]:
!pip install -q -U transformers accelerate pytest 2>&1 | tail -2
import transformers, torch
print('torch', torch.__version__, '| transformers', transformers.__version__)


## 3 · Correctness first

Do not measure a wrong answer. `run_tests.sh` runs one file per process — a single
pytest session holds ~5 model copies at once and will swap a small instance to a
standstill (see PROJECT_LOG.md B9).

Note the parity references in `results/mac/` were generated on MPS/float16. CUDA
kernels reduce differently, so the float16 tests may report drift — that is the
expected B1 behaviour, not a regression. The **float32** assertions are the ones
that must pass.


In [ ]:
!./run_tests.sh


## 4 · The six rungs

Every script takes `--device cuda` and writes to `results/gpu/` automatically.


In [ ]:
!python bench/run_baseline.py --device cuda --attn sdpa
!python bench/run_baseline.py --device cuda --attn eager   # parity reference


In [ ]:
!python bench/run_inferno.py --device cuda


### R2 — including the batch-16 anomaly

On MPS, throughput was flat from batch 1 to 8, **tripled at 16**, then fell back at 20 —
reproducible across three runs and unexplained (B5). The sweep below includes 12/16/20
specifically to see whether that survives on CUDA. If it does not, it was an MPS kernel
artifact and the MPS batching conclusions should be discarded.


In [ ]:
!python bench/run_batched.py --device cuda --batch-sizes 1,2,4,8,12,16,20,32


In [ ]:
!python bench/run_continuous.py --device cuda --pattern poisson --rates 1,2,4,8,16
!python bench/run_continuous.py --device cuda --pattern burst --rates 4


In [ ]:
!python bench/run_paged.py --device cuda --budget-mb 512 --n-requests 64
!python bench/run_paged.py --device cuda --budget-mb 512 --n-requests 128 --block-sizes 16


In [ ]:
!python bench/run_prefix.py --device cuda --workload rag --n-requests 15
!python bench/run_prefix.py --device cuda --workload mixed --n-requests 20


## 5 · vLLM

The comparison this whole phase exists for. Inferno is expected to lose; the value is in
knowing precisely where.

`bench/run_vllm.py` has **never been run** — it was written on a machine with no CUDA.
Treat the first execution as debugging, not measurement.


In [ ]:
!pip install -q vllm 2>&1 | tail -3


In [ ]:
import glob
ref = sorted(glob.glob('results/gpu/r0_baseline_*eager*.json'))[-1]
print('parity reference:', ref)
!python bench/run_vllm.py --reference $ref


Token agreement with Inferno will be **well below 100%**, and that is expected rather
than a defect: different kernels reduce in different orders and greedy decoding amplifies
the last bit. HuggingFace's own sdpa and eager paths disagreed on 10 of 50 prompts on the
same machine (B1). The number is reported, never asserted.


## 6 · Take the results home

Download the zip from the sidebar (Output → Files) before the session ends, then commit
it into `results/gpu/` locally. Kaggle deletes `/kaggle/working` when the session dies.


In [ ]:
!ls -la results/gpu/
!cd /kaggle/working/inferno && zip -qr /kaggle/working/inferno_gpu_results.zip results/gpu/
print('\nwrote /kaggle/working/inferno_gpu_results.zip')
!du -h /kaggle/working/inferno_gpu_results.zip


## 7 · What to look at first

1. **Did the batch-16 peak survive?** If not, B5 was an MPS artifact and the MPS batching
   conclusions go in the bin.
2. **Does batching finally scale 1→8?** On MPS it bought ~8%, against a theory predicting
   far more. A 0.5B model on a T4 may behave the same way for the same reason — the weights
   are small enough that decode is not weight-bandwidth-bound. If so, that is a real finding
   about model size, not about the engine.
3. **Paged vs contiguous at equal concurrency.** On MPS paging was ~8% *faster*, because
   both designs gather. Compare against vLLM here — that difference is the fused-kernel gap,
   and it is the single biggest unquantified item in the analysis.
4. **vLLM vs Inferno on decode throughput and TTFT.** Expect to lose. Write down by how much
   and, more importantly, why.
